In [1]:
import cv2
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import os, glob, time
from pathlib import Path

In [2]:
BASE_DIR   = r"D:\program vscode\MoneyLens\ai\Dataset_ocr"
ARRAYS_DIR = os.path.join(BASE_DIR, "preprocessed")
SPLITS     = ["train", "valid", "test"]
IMG_H, IMG_W = 32, 128
CHANNELS     = 1
 
CLASS_MAP = {
    0: "QTY",
    1: "harga_satuan",
    2: "nama_produk",
    3: "tanggal",
    4: "total_harga_barang",
    5: "total_transaksi",
}
 
# Prioritas kelas untuk MoneyLens
PRIORITAS = {
    "total_transaksi"  : "🔴 KRITIS  ",
    "tanggal"          : "🔴 KRITIS  ",
    "total_harga_barang": "🟡 PENTING ",
    "harga_satuan"     : "🟡 PENTING ",
    "QTY"              : "🟢 TAMBAHAN",
    "nama_produk"      : "🟢 TAMBAHAN",
}
 
CHARACTERS = list(
    "0123456789abcdefghijklmnopqrstuvwxyz"
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ .,:-/()%"
)
NUM_CLASSES = len(CHARACTERS) + 1

In [3]:
print("=" * 65)
print("TASK 3: EVALUASI PERFORMA PREPROCESSING PER KELAS")
print("=" * 65)
 
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU tersedia : {[g.name for g in gpus]}")
    print(f"Menggunakan  : GPU")
else:
    print(f"GPU          : Tidak tersedia")
    print(f"Menggunakan  : CPU")
print()

TASK 3: EVALUASI PERFORMA PREPROCESSING PER KELAS
GPU          : Tidak tersedia
Menggunakan  : CPU



In [4]:
def build_model():
    inputs = keras.Input(shape=(IMG_H, IMG_W, CHANNELS))
    x = layers.Conv2D(32, (3,3), padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2,1))(x)
    new_h = IMG_H // 8
    new_w = IMG_W // 4
    x = layers.Reshape((new_w, new_h * 128))(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(64,  return_sequences=True))(x)
    output = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, output)

In [5]:
task1_json = os.path.join(BASE_DIR, "task1_hasil.json")
if os.path.exists(task1_json):
    with open(task1_json) as f:
        task1 = json.load(f)
    print(f"Hasil Task 1  : {task1_json}")
    print(f"Rekomendasi   : {task1['nama_model']}")
    print(f"Arsitektur    : {task1['arsitektur']}")
    print(f"Device Task 1 : {task1['device']}")
else:
    print(f"[WARNING] task1_hasil.json tidak ditemukan!")
    print(f"          Jalankan Task 1 dulu.")
    task1 = {"rekomendasi": "v2", "arsitektur": "3CNN+2BiLSTM"}
print()
 
# Build model sesuai rekomendasi Task 1 (v2)
model = build_model()
print(f"Model        : {model.name}")
print(f"Params       : {model.count_params():,}")
print()

Hasil Task 1  : D:\program vscode\MoneyLens\ai\Dataset_ocr\task1_hasil.json
Rekomendasi   : OCR_v2_Sedang
Arsitektur    : 3CNN + 2BiLSTM + CTC
Device Task 1 : CPU

Model        : functional
Params       : 497,672



In [6]:
def pixel_contrast(arr: np.ndarray) -> float:
    """Kontras — semakin tinggi teks semakin jelas"""
    return round(float(arr.std()), 4)
 
def sharpness(arr: np.ndarray) -> float:
    """Ketajaman via Laplacian variance"""
    img_u8 = (arr[:,:,0] * 255).astype(np.uint8)
    return round(float(cv2.Laplacian(img_u8, cv2.CV_64F).var()), 2)
 
def black_pixel_ratio(arr: np.ndarray) -> float:
    """Rasio piksel teks — proxy kepadatan teks"""
    return round(float((arr < 0.5).mean()), 3)
 
def model_confidence(arr: np.ndarray) -> float:
    """Confidence model (forward pass tanpa training)"""
    batch = arr.reshape(1, IMG_H, IMG_W, CHANNELS)
    pred  = model.predict(batch, verbose=0)
    return round(float(pred.max(axis=-1).mean()) * 100, 2)
 
def inference_ms(arr: np.ndarray) -> float:
    """Kecepatan inferensi dalam milidetik"""
    batch = arr.reshape(1, IMG_H, IMG_W, CHANNELS)
    t0    = time.perf_counter()
    model.predict(batch, verbose=0)
    return round((time.perf_counter() - t0) * 1000, 1)

In [7]:
print(f"{'Kelas':<22} {'Prioritas':<12} {'N':>4} {'Contrast':>9} "
      f"{'Sharp':>8} {'BPR':>6} {'Conf%':>7} {'ms':>7}")
print("-" * 80)
 
all_results  = []
warmed_up    = False
 
for split in SPLITS:
    arrays_dir = os.path.join(ARRAYS_DIR, split, "arrays")
    if not os.path.exists(arrays_dir):
        print(f"[{split}] Folder tidak ditemukan, jalankan Task 2 dulu.")
        continue
 
    npy_files = glob.glob(os.path.join(arrays_dir, "*.npy"))
    if not npy_files:
        print(f"[{split}] Tidak ada .npy, jalankan Task 2 dulu.")
        continue
 
    for npy_path in npy_files:
        # Tentukan kelas dari nama file
        fname = Path(npy_path).stem
        cls   = None
        for class_name in PRIORITAS:
            if class_name in fname:
                cls = class_name
                break
        if cls is None:
            continue
 
        try:
            arr = np.load(npy_path)
            if arr.shape != (IMG_H, IMG_W, CHANNELS):
                continue
 
            # Warm-up GPU/CPU di iterasi pertama
            if not warmed_up:
                model.predict(
                    arr.reshape(1, IMG_H, IMG_W, CHANNELS), verbose=0)
                warmed_up = True
 
            contrast = pixel_contrast(arr)
            sharp    = sharpness(arr)
            bpr      = black_pixel_ratio(arr)
            conf     = model_confidence(arr)
            ms       = inference_ms(arr)
 
            all_results.append({
                "split"    : split,
                "file"     : fname,
                "kelas"    : cls,
                "prioritas": PRIORITAS[cls].strip(),
                "contrast" : contrast,
                "sharp"    : sharp,
                "bpr"      : bpr,
                "conf_pct" : conf,
                "ms"       : ms,
            })
 
        except Exception as e:
            print(f"  [ERROR] {Path(npy_path).name}: {e}")

Kelas                  Prioritas       N  Contrast    Sharp    BPR   Conf%      ms
--------------------------------------------------------------------------------


In [8]:
df = pd.DataFrame(all_results)
 
if df.empty:
    print("\n[WARNING] Tidak ada data .npy — jalankan Task 2 dulu!")
else:
    for cls, prio in PRIORITAS.items():
        rows = df[df["kelas"] == cls]
        if rows.empty:
            print(f"  {cls:<22} {prio:<12} {'—':>4}")
            continue
        print(f"  {cls:<22} {prio:<12} "
              f"{len(rows):>4} "
              f"{rows['contrast'].mean():>9.4f} "
              f"{rows['sharp'].mean():>8.2f} "
              f"{rows['bpr'].mean():>6.3f} "
              f"{rows['conf_pct'].mean():>6.1f}% "
              f"{rows['ms'].mean():>7.1f}")

  total_transaksi        🔴 KRITIS      401    0.2212  4203.55  0.098    1.5%   112.9
  tanggal                🔴 KRITIS      383    0.2648  6577.52  0.135    1.5%   112.4
  total_harga_barang     🟡 PENTING    1027    0.2596  5054.85  0.126    1.5%   113.9
  harga_satuan           🟡 PENTING     739    0.2644  5039.04  0.127    1.5%   112.4
  QTY                    🟢 TAMBAHAN    950    0.2551  3937.05  0.121    1.5%   113.3
  nama_produk            🟢 TAMBAHAN   1031    0.3193 15340.64  0.158    1.5%   114.0


In [11]:
print(f"\n{'='*65}")
print("ANALISIS KELAS KRITIS (MoneyLens)")
print(f"{'='*65}")
 
for cls in ["total_transaksi", "tanggal"]:
    rows = df[df["kelas"] == cls]
    if rows.empty:
        print(f"\n  {cls}: tidak ada data")
        continue
    avg_sharp = rows["sharp"].mean()
    avg_conf  = rows["conf_pct"].mean()
    avg_bpr   = rows["bpr"].mean()
    status    = "✅ LAYAK" if avg_sharp > 50 else "⚠️  PERLU PERBAIKAN"

    print(f"\n  {cls}:")
    print(f"    Jumlah sampel  : {len(rows)}")
    print(f"    Avg Sharpness  : {avg_sharp:.2f}")
    print(f"    Avg Confidence : {avg_conf:.1f}%")
    print(f"    Avg BPR        : {avg_bpr:.3f}")
    print(f"    Status         : {status}")

    if avg_bpr < 0.05:
        print(f"    ⚠️  BPR rendah → teks mungkin terlalu tipis/hilang")
    elif avg_bpr > 0.6:
        print(f"    ⚠️  BPR tinggi → terlalu banyak noise")


ANALISIS KELAS KRITIS (MoneyLens)

  total_transaksi:
    Jumlah sampel  : 401
    Avg Sharpness  : 4203.55
    Avg Confidence : 1.5%
    Avg BPR        : 0.098
    Status         : ✅ LAYAK

  tanggal:
    Jumlah sampel  : 383
    Avg Sharpness  : 6577.52
    Avg Confidence : 1.5%
    Avg BPR        : 0.135
    Status         : ✅ LAYAK


In [13]:
out_csv = os.path.join(BASE_DIR, "task3_evaluasi_awal.csv")
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n💾 Hasil disimpan: {out_csv}")
 
print(f"\n{'='*65}")
print("KESIMPULAN")
print(f"{'='*65}")
print("  Fokus utama MoneyLens:")
print("    → total_transaksi : nominal yang diinput user")
print("    → tanggal         : tanggal transaksi")
print()
print("  Jika sharpness kelas kritis < 50:")
print("    → Naikkan PADDING di Task 2")
print("    → Cek apakah gambar dari teman sudah bersih")
print()
print("  Next step setelah eksperimen:")
print("    → Labeling teks ground truth per crop")
print("    → Training model dengan tf.GradientTape")
print(f"{'='*65}")


💾 Hasil disimpan: D:\program vscode\MoneyLens\ai\Dataset_ocr\task3_evaluasi_awal.csv

KESIMPULAN
  Fokus utama MoneyLens:
    → total_transaksi : nominal yang diinput user
    → tanggal         : tanggal transaksi

  Jika sharpness kelas kritis < 50:
    → Naikkan PADDING di Task 2
    → Cek apakah gambar dari teman sudah bersih

  Next step setelah eksperimen:
    → Labeling teks ground truth per crop
    → Training model dengan tf.GradientTape
